# Limpeza e diagnóstico — ENEM 2023 (Matemática)

Este notebook documenta a seleção das colunas, o tratamento de registros sem nota-alvo, a investigação de possíveis valores extremos e a codificação numérica das features. A base bruta permanece intacta; as leituras e exportações são feitas em blocos para reduzir o uso de memória.

**Regras:** manter os participantes treineiros, excluir da regressão apenas registros sem `NU_NOTA_MT`, não remover automaticamente valores extremos e gerar duas saídas: uma com os códigos originais e outra numérica para modelagem.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RAIZ_PROJETO = Path.cwd()
if not (RAIZ_PROJETO / "dados").exists() and (RAIZ_PROJETO.parent / "dados").exists():
    RAIZ_PROJETO = RAIZ_PROJETO.parent

caminhos_csv = [
    RAIZ_PROJETO / "dados" / "raw" / "MICRODADOS_ENEM_2023.csv",
    RAIZ_PROJETO / "dados" / "raw" / "DADOS" / "MICRODADOS_ENEM_2023.csv",
]
CAMINHO_CSV = next((p for p in caminhos_csv if p.exists()), None)
if CAMINHO_CSV is None:
    raise FileNotFoundError("MICRODADOS_ENEM_2023.csv não encontrado em dados/raw/.")

COLUNA_ID = "NU_INSCRICAO"
ALVO = "NU_NOTA_MT"
TAMANHO_BLOCO = 100_000

# Features escolhidas pela equipe (33): perfil/trajetória escolar + Q001–Q025.
FEATURES_MANTIDAS = [
    "TP_FAIXA_ETARIA", "TP_SEXO", "TP_ESTADO_CIVIL", "TP_COR_RACA",
    "TP_ST_CONCLUSAO", "TP_ESCOLA", "TP_ENSINO", "IN_TREINEIRO",
    *[f"Q{i:03d}" for i in range(1, 26)],
]

# As 46 colunas candidatas registradas na tabela de decisões da equipe.
FEATURES_CANDIDATAS = [
    "TP_FAIXA_ETARIA", "TP_SEXO", "TP_ESTADO_CIVIL", "TP_COR_RACA",
    "TP_NACIONALIDADE", "TP_ST_CONCLUSAO", "TP_ANO_CONCLUIU", "TP_ESCOLA",
    "TP_ENSINO", "IN_TREINEIRO", "CO_MUNICIPIO_ESC", "NO_MUNICIPIO_ESC",
    "CO_UF_ESC", "SG_UF_ESC", "TP_DEPENDENCIA_ADM_ESC", "TP_LOCALIZACAO_ESC",
    "TP_SIT_FUNC_ESC", "CO_MUNICIPIO_PROVA", "NO_MUNICIPIO_PROVA",
    "CO_UF_PROVA", "SG_UF_PROVA", *[f"Q{i:03d}" for i in range(1, 26)],
]
FEATURES_REMOVIDAS_DECISAO = [
    coluna for coluna in FEATURES_CANDIDATAS if coluna not in FEATURES_MANTIDAS
]
COLUNAS_SAIDA_LIMPA = [COLUNA_ID, *FEATURES_MANTIDAS, ALVO]
COLUNAS_DIAGNOSTICO = [COLUNA_ID, *FEATURES_CANDIDATAS, ALVO]

colunas_csv = pd.read_csv(CAMINHO_CSV, sep=";", encoding="latin1", nrows=0).columns.tolist()
colunas_ausentes_no_csv = sorted(set(COLUNAS_DIAGNOSTICO) - set(colunas_csv))
if colunas_ausentes_no_csv:
    raise KeyError(f"Colunas esperadas não encontradas na base: {colunas_ausentes_no_csv}")
if len(FEATURES_MANTIDAS) != 33 or len(FEATURES_REMOVIDAS_DECISAO) != 13:
    raise AssertionError("A seleção de features mudou; revise as listas e a tabela de decisões.")

print(f"Arquivo bruto: {CAMINHO_CSV}")
print(f"Colunas na base original: {len(colunas_csv)}")
print(f"Features candidatas da equipe: {len(FEATURES_CANDIDATAS)}")
print(f"Features mantidas em X ({len(FEATURES_MANTIDAS)}):")
for coluna in FEATURES_MANTIDAS:
    print(f"  + {coluna}")
print(f"\nFeatures removidas das candidatas ({len(FEATURES_REMOVIDAS_DECISAO)}):")
for coluna in FEATURES_REMOVIDAS_DECISAO:
    print(f"  - {coluna}")
print("\nFeature adicional fora de X por decisão anterior: TP_LINGUA")
print(f"Colunas na saída (ID de controle + X + y): {len(COLUNAS_SAIDA_LIMPA)}")
print(f"Outras colunas brutas não exportadas: {len(colunas_csv) - len(COLUNAS_SAIDA_LIMPA)}")


Arquivo bruto: /Users/joaoigorandrade/Documents/unifor/cienciadedados/ciencia-de-dados-unifor-projeto-2/dados/raw/DADOS/MICRODADOS_ENEM_2023.csv
Colunas na base original: 76
Features candidatas da equipe: 46
Features mantidas em X (33):
  + TP_FAIXA_ETARIA
  + TP_SEXO
  + TP_ESTADO_CIVIL
  + TP_COR_RACA
  + TP_ST_CONCLUSAO
  + TP_ESCOLA
  + TP_ENSINO
  + IN_TREINEIRO
  + Q001
  + Q002
  + Q003
  + Q004
  + Q005
  + Q006
  + Q007
  + Q008
  + Q009
  + Q010
  + Q011
  + Q012
  + Q013
  + Q014
  + Q015
  + Q016
  + Q017
  + Q018
  + Q019
  + Q020
  + Q021
  + Q022
  + Q023
  + Q024
  + Q025

Features removidas das candidatas (13):
  - TP_NACIONALIDADE
  - TP_ANO_CONCLUIU
  - CO_MUNICIPIO_ESC
  - NO_MUNICIPIO_ESC
  - CO_UF_ESC
  - SG_UF_ESC
  - TP_DEPENDENCIA_ADM_ESC
  - TP_LOCALIZACAO_ESC
  - TP_SIT_FUNC_ESC
  - CO_MUNICIPIO_PROVA
  - NO_MUNICIPIO_PROVA
  - CO_UF_PROVA
  - SG_UF_PROVA

Feature adicional fora de X por decisão anterior: TP_LINGUA
Colunas na saída (ID de controle + X + y): 3

## O que será tratado — e o que não será

- **Colunas:** manter somente as 33 features escolhidas, `NU_NOTA_MT` (alvo) e `NU_INSCRICAO` (controle). Das 46 candidatas da tabela, 13 ficam fora de `X`. A coluna de inscrição não pode entrar no modelo.
- **Alvo ausente:** esses registros entram no diagnóstico, mas não podem participar da regressão supervisionada. Não imputar a nota nem convertê-la em zero.
- **Treineiros:** manter. A coluna `IN_TREINEIRO` é permitida como feature; não excluir linhas por seu valor.
- **Ausências nas features:** manter como ausentes nesta etapa. O tratamento deve ser decidido e ajustado dentro do pipeline/treino de cada dobra, sem aprender imputações usando validação ou teste. `TP_ENSINO` requer atenção especial por sua alta proporção de ausências.
- **Outliers:** identificar estatisticamente, investigar e preservar se forem valores válidos. Códigos categóricos não são medidas contínuas; não aplicar IQR aos códigos como se fossem quantidades.


In [2]:
# Uma passagem pela base completa: contagens de ausências, alvo, treineiros e notas observadas.
contagem_linhas = 0
contagem_alvo_ausente = 0
contagem_treineiros = 0
contagem_treineiros_com_nota = 0
contagem_treineiros_sem_nota = 0
contagem_id_ausente = 0
ausencias_features = {coluna: 0 for coluna in FEATURES_CANDIDATAS}
ausencias_features_com_nota = {coluna: 0 for coluna in FEATURES_CANDIDATAS}
blocos_notas = []

leitor = pd.read_csv(
    CAMINHO_CSV,
    sep=";",
    encoding="latin1",
    usecols=COLUNAS_DIAGNOSTICO,
    dtype={coluna: "string" for coluna in [COLUNA_ID, *FEATURES_CANDIDATAS]},
    chunksize=TAMANHO_BLOCO,
)

for numero_bloco, bloco in enumerate(leitor, start=1):
    contagem_linhas += len(bloco)
    alvo_observado = bloco[ALVO].notna()
    contagem_alvo_ausente += int((~alvo_observado).sum())
    contagem_id_ausente += int(bloco[COLUNA_ID].isna().sum())

    eh_treineiro = bloco["IN_TREINEIRO"].eq("1").fillna(False)
    contagem_treineiros += int(eh_treineiro.sum())
    contagem_treineiros_com_nota += int((eh_treineiro & alvo_observado).sum())
    contagem_treineiros_sem_nota += int((eh_treineiro & ~alvo_observado).sum())

    for coluna in FEATURES_CANDIDATAS:
        ausente = bloco[coluna].isna()
        ausencias_features[coluna] += int(ausente.sum())
        ausencias_features_com_nota[coluna] += int((ausente & alvo_observado).sum())

    notas = pd.to_numeric(bloco.loc[alvo_observado, ALVO], errors="coerce").dropna()
    blocos_notas.append(notas.to_numpy(dtype="float64"))

    if numero_bloco % 10 == 0:
        print(f"Blocos lidos: {numero_bloco} | linhas examinadas: {contagem_linhas:,}")

notas_mt = np.concatenate(blocos_notas) if blocos_notas else np.array([], dtype="float64")
registros_com_nota = len(notas_mt)
registros_sem_nota = contagem_linhas - registros_com_nota

resumo_limpeza = pd.DataFrame([
    {"indicador": "Registros na base bruta", "quantidade": contagem_linhas},
    {"indicador": "Registros sem NU_NOTA_MT (fora da regressão)", "quantidade": registros_sem_nota},
    {"indicador": "Registros com NU_NOTA_MT (elegíveis)", "quantidade": registros_com_nota},
    {"indicador": "Treineiros na base bruta", "quantidade": contagem_treineiros},
    {"indicador": "Treineiros elegíveis mantidos (nota observada)", "quantidade": contagem_treineiros_com_nota},
    {"indicador": "Treineiros sem nota-alvo (não entram na regressão)", "quantidade": contagem_treineiros_sem_nota},
    {"indicador": "NU_INSCRICAO ausente", "quantidade": contagem_id_ausente},
])
display(resumo_limpeza)

linhas_ausencias = []
for coluna in FEATURES_CANDIDATAS:
    linhas_ausencias.append({
        "feature": coluna,
        "ausentes_base": ausencias_features[coluna],
        "percentual_ausente_base": round(ausencias_features[coluna] / contagem_linhas * 100, 2),
        "ausentes_com_nota_alvo": ausencias_features_com_nota[coluna],
        "percentual_ausente_com_nota": round(ausencias_features_com_nota[coluna] / registros_com_nota * 100, 2),
    })
df_ausencias_features = pd.DataFrame(linhas_ausencias).sort_values(
    "percentual_ausente_base", ascending=False
).reset_index(drop=True)
print("Features candidatas com pelo menos uma ausência real:")
display(df_ausencias_features.loc[df_ausencias_features["ausentes_base"] > 0])


Blocos lidos: 10 | linhas examinadas: 1,000,000
Blocos lidos: 20 | linhas examinadas: 2,000,000
Blocos lidos: 30 | linhas examinadas: 3,000,000
Blocos lidos: 40 | linhas examinadas: 3,933,955


,indicador,quantidade
0,Registros na base bruta,3933955
1,Registros sem NU_NOTA_MT (fora da regressão),1241528
2,Registros com NU_NOTA_MT (elegíveis),2692427
3,Treineiros na base bruta,620067
4,Treineiros elegíveis mantidos (nota observada),516920
5,Treineiros sem nota-alvo (não entram na regres...,103147
6,NU_INSCRICAO ausente,0


Features candidatas com pelo menos uma ausência real:


,feature,ausentes_base,percentual_ausente_base,ausentes_com_nota_alvo,percentual_ausente_com_nota
0,SG_UF_ESC,2975449,75.64,1968042,73.10
1,CO_MUNICIPIO_ESC,2975449,75.64,1968042,73.10
2,TP_LOCALIZACAO_ESC,2975449,75.64,1968042,73.10
3,TP_DEPENDENCIA_ADM_ESC,2975449,75.64,1968042,73.10
4,CO_UF_ESC,2975449,75.64,1968042,73.10
5,NO_MUNICIPIO_ESC,2975449,75.64,1968042,73.10
6,TP_SIT_FUNC_ESC,2975449,75.64,1968042,73.10
7,TP_ENSINO,2594874,65.96,1660064,61.66


## Investigação de outliers na nota de Matemática

A nota é contínua, então calculamos os limites usuais de 1,5 × IQR como **sinalizador exploratório**, não como regra automática de exclusão. Nota zero é um resultado observado e válido. Valores acima do limite superior também podem ser notas legítimas. Como referência de validade, verificamos ainda se há notas fora da escala de 0 a 1000.


In [3]:
if len(notas_mt) == 0:
    raise ValueError("Não há notas observadas para analisar.")

q1, mediana, q3 = np.quantile(notas_mt, [0.25, 0.50, 0.75])
iqr = q3 - q1
limite_inferior_iqr = q1 - 1.5 * iqr
limite_superior_iqr = q3 + 1.5 * iqr
mascara_inferior = notas_mt < limite_inferior_iqr
mascara_superior = notas_mt > limite_superior_iqr
mascara_fora_escala = (notas_mt < 0) | (notas_mt > 1000)

resumo_outliers = pd.DataFrame([
    {"medida": "Notas observadas", "valor": len(notas_mt)},
    {"medida": "Nota mínima", "valor": float(notas_mt.min())},
    {"medida": "Nota máxima", "valor": float(notas_mt.max())},
    {"medida": "Q1", "valor": float(q1)},
    {"medida": "Mediana", "valor": float(mediana)},
    {"medida": "Q3", "valor": float(q3)},
    {"medida": "IQR", "valor": float(iqr)},
    {"medida": "Limite inferior IQR", "valor": float(limite_inferior_iqr)},
    {"medida": "Limite superior IQR", "valor": float(limite_superior_iqr)},
    {"medida": "Candidatos abaixo do limite IQR", "valor": int(mascara_inferior.sum())},
    {"medida": "Desses, notas zero (válidas)", "valor": int((notas_mt[mascara_inferior] == 0).sum())},
    {"medida": "Candidatos acima do limite IQR", "valor": int(mascara_superior.sum())},
    {"medida": "Candidatos fora da escala 0–1000", "valor": int(mascara_fora_escala.sum())},
])
display(resumo_outliers)
print("Decisão: nenhum ponto será removido apenas por ser sinalizado pelo critério IQR.")

# Histograma agregado: mostra a distribuição sem enviar milhões de pontos ao gráfico.
import plotly.graph_objects as go

bins_nota = np.arange(0, 1000 + 20, 20)
contagens_histograma, bordas_histograma = np.histogram(notas_mt, bins=bins_nota)
rotulos_intervalos = [
    f"{int(inicio)}–{int(fim)}" for inicio, fim in zip(bordas_histograma[:-1], bordas_histograma[1:])
]
fig_nota = go.Figure(go.Bar(x=rotulos_intervalos, y=contagens_histograma, marker_color="#4C78A8"))
fig_nota.update_layout(
    title="Distribuição de NU_NOTA_MT — participantes com nota observada",
    xaxis_title="Intervalo de nota",
    yaxis_title="Quantidade de participantes",
    xaxis_tickangle=-45,
    template="plotly_white",
)
fig_nota.show()


,medida,valor
0,Notas observadas,2692427.00
1,Nota mínima,0.00
2,Nota máxima,958.60
3,Q1,431.20
4,Mediana,523.60
5,Q3,630.10
6,IQR,198.90
7,Limite inferior IQR,132.85
8,Limite superior IQR,928.45
9,Candidatos abaixo do limite IQR,16638.00


Decisão: nenhum ponto será removido apenas por ser sinalizado pelo critério IQR.


## Duplicatas pelo identificador

`NU_INSCRICAO` é uma chave de controle, não uma feature. A célula seguinte verifica valores ausentes e inscrições repetidas lendo apenas essa coluna. Se houvesse repetição, a decisão seria investigar antes de remover linhas.


In [4]:
ids_inscricao = pd.read_csv(
    CAMINHO_CSV,
    sep=";",
    encoding="latin1",
    usecols=[COLUNA_ID],
)[COLUNA_ID]

resumo_ids = pd.DataFrame([{
    "registros": len(ids_inscricao),
    "IDs ausentes": int(ids_inscricao.isna().sum()),
    "IDs duplicados após a primeira ocorrência": int(ids_inscricao.duplicated().sum()),
    "IDs distintos": int(ids_inscricao.nunique(dropna=True)),
}])
display(resumo_ids)
del ids_inscricao


,registros,IDs ausentes,IDs duplicados após a primeira ocorrência,IDs distintos
0,3933955,0,0,3933955


## Resultado observado na base local

Na execução de diagnóstico desta base, foram encontrados **3.933.955 registros**; **1.241.528** não têm `NU_NOTA_MT` e não entram na regressão; **2.692.427** têm nota observada. Desses, **516.920 treineiros** têm nota observada e permanecem elegíveis. Nenhum participante é removido por ser treineiro.

Pelo critério IQR, os limites observados foram aproximadamente **132,85** e **928,45**. Os **16.638** candidatos inferiores eram todos notas zero, que são válidas; **1.904** notas ficaram acima do limite superior. A faixa observada foi **0 a 958,6**, sem valores fora da escala 0–1000. Portanto, os pontos sinalizados serão mantidos e investigados, não apagados automaticamente.

A verificação de `NU_INSCRICAO` encontrou **0 IDs ausentes e 0 duplicados**. Entre as features candidatas, as ausências concentraram-se nas colunas de escola (75,64% na base completa) e em `TP_ENSINO` (65,96%). Reexecute as células para atualizar esses resultados caso a base mude.


A exportação cria a versão categórica `base_modelagem_mt_limpa.csv` e a versão exclusivamente numérica `base_modelagem_mt_numerica.csv`, ambas em `dados/processed/DADOS/`.

## Exportação das bases de modelagem

A primeira exportação salva `dados/processed/DADOS/base_modelagem_mt_limpa.csv`, com ID de controle, 33 features e alvo, preservando os códigos categóricos originais. A última célula também salva `dados/processed/DADOS/base_modelagem_mt_numerica.csv`, em que as variáveis nominais viram colunas indicadoras 0/1 e as variáveis ordinais/contagens usam mapeamentos documentados.

Ambas removem somente registros sem nota-alvo. Treineiros, notas zero, candidatos a outlier e ausências nas features são preservados. Não há imputação nem remoção automática de outliers. Se um arquivo já existir, ele não será sobrescrito.

In [5]:
def exportar_base_limpa(caminho_saida=None):
    """Exporta somente ID, features aprovadas e alvo; mantém treineiros e notas zero."""
    pasta_processados = RAIZ_PROJETO / "dados" / "processed" / "DADOS"
    pasta_processados.mkdir(parents=True, exist_ok=True)
    if caminho_saida is None:
        caminho_saida = pasta_processados / "base_modelagem_mt_limpa.csv"
    else:
        caminho_saida = Path(caminho_saida)
        caminho_saida.parent.mkdir(parents=True, exist_ok=True)
    temporario = caminho_saida.with_name(caminho_saida.name + ".tmp")
    if caminho_saida.exists():
        print(f"O arquivo já existe; não foi sobrescrito: {caminho_saida}")
        print(f"Features mantidas: {len(FEATURES_MANTIDAS)} | candidatas removidas: {len(FEATURES_REMOVIDAS_DECISAO)}")
        print(f"Registros mantidos (nota observada): {registros_com_nota:,}")
        print(f"Treineiros elegíveis mantidos: {contagem_treineiros_com_nota:,}")
        return caminho_saida
    if temporario.exists():
        raise FileExistsError(f"Arquivo temporário já existe: {temporario}")

    dtype_texto = {coluna: "string" for coluna in [COLUNA_ID, *FEATURES_MANTIDAS]}
    total_lido = 0
    total_sem_alvo = 0
    total_mantido = 0
    treineiros_mantidos = 0
    outliers_iqr_mantidos = 0
    primeiro_bloco = True
    try:
        for bloco in pd.read_csv(
            CAMINHO_CSV,
            sep=";",
            encoding="latin1",
            usecols=COLUNAS_SAIDA_LIMPA,
            dtype=dtype_texto,
            chunksize=TAMANHO_BLOCO,
        ):
            bloco = bloco[COLUNAS_SAIDA_LIMPA]
            total_lido += len(bloco)
            valido = bloco[ALVO].notna()
            total_sem_alvo += int((~valido).sum())
            limpo = bloco.loc[valido].copy()
            total_mantido += len(limpo)
            treineiros_mantidos += int(limpo["IN_TREINEIRO"].eq("1").sum())
            notas_bloco = pd.to_numeric(limpo[ALVO], errors="coerce")
            outliers_iqr_mantidos += int(
                ((notas_bloco < limite_inferior_iqr) | (notas_bloco > limite_superior_iqr)).sum()
            )
            limpo.to_csv(temporario, mode="w" if primeiro_bloco else "a", header=primeiro_bloco, index=False)
            primeiro_bloco = False
    except Exception:
        if temporario.exists():
            temporario.unlink()
        raise

    temporario.replace(caminho_saida)
    print(f"Colunas da base bruta: {len(colunas_csv)}")
    print(f"Features candidatas removidas de X: {len(FEATURES_REMOVIDAS_DECISAO)}")
    print(f"Colunas não exportadas além de ID, X e y: {len(colunas_csv) - len(COLUNAS_SAIDA_LIMPA)}")
    print(f"Registros lidos: {total_lido:,}")
    print(f"Registros removidos por NU_NOTA_MT ausente: {total_sem_alvo:,}")
    print(f"Registros mantidos: {total_mantido:,}")
    print(f"Treineiros mantidos entre os elegíveis: {treineiros_mantidos:,}")
    print(f"Candidatos estatísticos a outlier mantidos (IQR): {outliers_iqr_mantidos:,}")
    print(f"Treineiros removidos por serem treineiros: 0")
    print(f"Arquivo limpo salvo em: {caminho_saida}")
    return caminho_saida

caminho_base_limpa = exportar_base_limpa()


O arquivo já existe; não foi sobrescrito: /Users/joaoigorandrade/Documents/unifor/cienciadedados/ciencia-de-dados-unifor-projeto-2/dados/processed/DADOS/base_modelagem_mt_limpa.csv
Features mantidas: 33 | candidatas removidas: 13
Registros mantidos (nota observada): 2,692,427
Treineiros elegíveis mantidos: 516,920


## Codificação numérica das respostas

Não substituímos letras por números arbitrários em todas as perguntas. Sexo/raça/estado civil/escola e ocupação são nominais: usam dummies numéricas 0/1 com categorias fixas do dicionário e uma referência. Escolaridade e renda têm ordem: recebem códigos/estimativas documentadas. Contagens limitadas usam o piso da categoria (`4 ou mais` → 4). `H` em Q001/Q002 é “não sei”, separado de ausência. Campos ausentes permanecem ausentes e ganham indicador; não são imputados aqui.

A análise de outliers da nota continua no notebook: os sinalizados pelo IQR, incluindo zeros válidos, são preservados; apenas nota-alvo ausente exclui uma linha da regressão. Nas features, a verificação apropriada é contra os domínios do dicionário (códigos inesperados); IQR não deve ser aplicado a dummies nem a categorias numéricas como se fossem medidas contínuas.


In [6]:
# Codificação determinística, baseada no dicionário do ENEM 2023.
# Categorias nominais são one-hot; escalas ordenadas/contagens usam mapeamentos documentados.
CATEGORIAS_NOMINAIS = {
    "TP_FAIXA_ETARIA": [str(i) for i in range(1, 21)],
    "TP_SEXO": ["F", "M"],
    "TP_ESTADO_CIVIL": [str(i) for i in range(0, 5)],
    "TP_COR_RACA": [str(i) for i in range(0, 7)],
    "TP_ST_CONCLUSAO": [str(i) for i in range(1, 5)],
    "TP_ESCOLA": [str(i) for i in range(1, 4)],
    "TP_ENSINO": ["1", "2"],
    "Q003": list("ABCDEF"),
    "Q004": list("ABCDEF"),
}

MAPA_ESCOLARIDADE = {letra: nivel for nivel, letra in enumerate("ABCDEFG")}
MAPA_RENDA_SM = {
    "A": 0.0, "B": 0.5, "C": 1.25, "D": 1.75, "E": 2.25, "F": 2.75,
    "G": 3.5, "H": 4.5, "I": 5.5, "J": 6.5, "K": 7.5, "L": 8.5,
    "M": 9.5, "N": 11.0, "O": 13.5, "P": 17.5, "Q": 20.0,
}
MAPA_CONTAGEM = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}  # E = 4 ou mais (piso)
MAPA_DIAS_DOMESTICA = {"A": 0, "B": 1, "C": 2, "D": 3}  # D representa 3 ou mais dias/semana
COLUNAS_CONTAGEM = ["Q008", "Q009", "Q010", "Q011", "Q012", "Q013", "Q014", "Q015", "Q016", "Q017", "Q019", "Q022", "Q024"]
COLUNAS_BINARIAS = ["Q018", "Q020", "Q021", "Q023", "Q025"]


def normalizar_codigo(serie):
    """Normaliza códigos como 1, 1.0 ou letras, sem interpretar o código como medida."""
    return serie.astype("string").str.strip().str.replace(r"\.0+$", "", regex=True)


def codificar_categoria_nominal(saida, coluna, serie, categorias):
    valores = normalizar_codigo(serie)
    # A primeira categoria do dicionário é a referência (todas as dummies = 0).
    for categoria in categorias[1:]:
        saida[f"{coluna}_{categoria}"] = valores.eq(categoria).fillna(False).astype("int8")
    saida[f"{coluna}__AUSENTE"] = valores.isna().astype("int8")
    saida[f"{coluna}__OUTRO"] = (valores.notna() & ~valores.isin(categorias)).astype("int8")


def codificar_valor_ordenado(colunas_numericas, coluna_saida, serie, mapa, codigos_validos):
    valores = normalizar_codigo(serie)
    colunas_numericas[coluna_saida] = valores.map(mapa).astype("float32")
    colunas_numericas[f"{coluna_saida}__AUSENTE"] = valores.isna().astype("int8")
    colunas_numericas[f"{coluna_saida}__OUTRO"] = (valores.notna() & ~valores.isin(codigos_validos)).astype("int8")


def transformar_bloco_para_numerico(bloco):
    """Devolve ID + preditoras numéricas + alvo; nenhum rótulo textual é exportado."""
    colunas_numericas = {}
    id_numerico = pd.to_numeric(bloco[COLUNA_ID], errors="coerce")
    if id_numerico.isna().any():
        raise ValueError("Há NU_INSCRICAO ausente ou não numérica; verificar a base antes de exportar.")
    colunas_numericas[COLUNA_ID] = id_numerico.astype("int64")

    for coluna, categorias in CATEGORIAS_NOMINAIS.items():
        codificar_categoria_nominal(colunas_numericas, coluna, bloco[coluna], categorias)

    # Escolaridade: A–G são níveis ordenados; H ('não sei') é sinalizado à parte,
    # não tratado como um nível superior. Ausência real e código inesperado ficam separados.
    for coluna in ("Q001", "Q002"):
        valores = normalizar_codigo(bloco[coluna])
        colunas_numericas[f"{coluna}_NIVEL_ESCOLARIDADE"] = valores.map(MAPA_ESCOLARIDADE).astype("float32")
        colunas_numericas[f"{coluna}_DESCONHECE"] = valores.eq("H").fillna(False).astype("int8")
        colunas_numericas[f"{coluna}__AUSENTE"] = valores.isna().astype("int8")
        colunas_numericas[f"{coluna}__OUTRO"] = (valores.notna() & ~valores.isin(list("ABCDEFGH"))).astype("int8")

    # Condição binária de treineiro é preservada como 0/1; nenhuma linha é filtrada por ela.
    codificar_valor_ordenado(colunas_numericas, "IN_TREINEIRO", bloco["IN_TREINEIRO"], {"0": 0, "1": 1}, ["0", "1"])

    # Contagens quantitativas já numéricas no questionário.
    moradores = pd.to_numeric(normalizar_codigo(bloco["Q005"]), errors="coerce")
    valido_moradores = moradores.between(1, 20)
    colunas_numericas["Q005_MORADORES"] = moradores.where(valido_moradores).astype("float32")
    valores_moradores = normalizar_codigo(bloco["Q005"])
    colunas_numericas["Q005_MORADORES__AUSENTE"] = valores_moradores.isna().astype("int8")
    colunas_numericas["Q005_MORADORES__FORA_DOMINIO"] = (valores_moradores.notna() & ~valido_moradores).astype("int8")

    # Renda familiar: ponto médio aproximado das faixas em salários mínimos;
    # Q corresponde a mais de 20 SM e recebe 20 como limite inferior.
    codificar_valor_ordenado(colunas_numericas, "Q006_RENDA_SM_ESTIMADA", bloco["Q006"], MAPA_RENDA_SM, list(MAPA_RENDA_SM))
    codificar_valor_ordenado(colunas_numericas, "Q007_DIAS_DOMESTICA_ESTIMADOS", bloco["Q007"], MAPA_DIAS_DOMESTICA, list(MAPA_DIAS_DOMESTICA))

    for coluna in COLUNAS_CONTAGEM:
        codificar_valor_ordenado(colunas_numericas, f"{coluna}_CONTAGEM_ESTIMADA", bloco[coluna], MAPA_CONTAGEM, list(MAPA_CONTAGEM))
    for coluna in COLUNAS_BINARIAS:
        codificar_valor_ordenado(colunas_numericas, f"{coluna}_BINARIA", bloco[coluna], {"A": 0, "B": 1}, ["A", "B"])

    colunas_numericas[ALVO] = pd.to_numeric(bloco[ALVO], errors="coerce").astype("float32")
    saida = pd.DataFrame(colunas_numericas, index=bloco.index)
    if not all(pd.api.types.is_numeric_dtype(tipo) for tipo in saida.dtypes):
        raise TypeError("A saída numérica contém uma coluna não numérica.")
    return saida


def exportar_base_numerica(caminho_saida=None):
    """Codifica em blocos e salva uma versão numérica; não sobrescreve arquivo existente."""
    pasta_processados = RAIZ_PROJETO / "dados" / "processed" / "DADOS"
    pasta_processados.mkdir(parents=True, exist_ok=True)
    if caminho_saida is None:
        caminho_saida = pasta_processados / "base_modelagem_mt_numerica.csv"
    else:
        caminho_saida = Path(caminho_saida)
        caminho_saida.parent.mkdir(parents=True, exist_ok=True)
    temporario = caminho_saida.with_name(caminho_saida.name + ".tmp")
    if caminho_saida.exists():
        print(f"O arquivo numérico já existe; não foi sobrescrito: {caminho_saida}")
        return caminho_saida
    if temporario.exists():
        raise FileExistsError(f"Arquivo temporário já existe: {temporario}")

    dtype_texto = {coluna: "string" for coluna in [COLUNA_ID, *FEATURES_MANTIDAS]}
    total_lido = total_sem_alvo = total_mantido = treineiros_mantidos = 0
    zeros_mantidos = outliers_iqr_mantidos = 0
    codigos_fora_dominio = {}
    primeiro_bloco = True
    try:
        leitor = pd.read_csv(
            CAMINHO_CSV,
            sep=";",
            encoding="latin1",
            usecols=COLUNAS_SAIDA_LIMPA,
            dtype=dtype_texto,
            chunksize=TAMANHO_BLOCO,
        )
        for bloco in leitor:
            total_lido += len(bloco)
            alvo = pd.to_numeric(bloco[ALVO], errors="coerce")
            valido = alvo.notna()
            total_sem_alvo += int((~valido).sum())
            bloco_valido = bloco.loc[valido].copy()
            total_mantido += len(bloco_valido)
            treineiros_mantidos += int(normalizar_codigo(bloco_valido["IN_TREINEIRO"]).eq("1").sum())
            notas_validas = pd.to_numeric(bloco_valido[ALVO], errors="coerce")
            zeros_mantidos += int(notas_validas.eq(0).sum())
            outliers_iqr_mantidos += int(
                ((notas_validas < limite_inferior_iqr) | (notas_validas > limite_superior_iqr)).sum()
            )
            numerico = transformar_bloco_para_numerico(bloco_valido)
            for coluna_diagnostico in numerico.columns:
                if coluna_diagnostico.endswith("__OUTRO") or coluna_diagnostico.endswith("__FORA_DOMINIO"):
                    quantidade_invalida = int(numerico[coluna_diagnostico].sum())
                    if quantidade_invalida:
                        codigos_fora_dominio[coluna_diagnostico] = codigos_fora_dominio.get(coluna_diagnostico, 0) + quantidade_invalida
            numerico.to_csv(temporario, mode="w" if primeiro_bloco else "a", header=primeiro_bloco, index=False)
            primeiro_bloco = False
    except Exception:
        if temporario.exists():
            temporario.unlink()
        raise

    temporario.replace(caminho_saida)
    print(f"Features nominais e ordinais transformadas em colunas/valores numéricos: {len(FEATURES_MANTIDAS)} features de origem.")
    print(f"Features candidatas excluídas de X: {len(FEATURES_REMOVIDAS_DECISAO)} | TP_LINGUA também excluída.")
    print(f"Colunas numéricas na saída (incluindo ID e y): {len(numerico.columns)}")
    print(f"Registros lidos: {total_lido:,} | removidos por alvo ausente: {total_sem_alvo:,} | mantidos: {total_mantido:,}")
    print(f"Treineiros mantidos: {treineiros_mantidos:,} | notas zero mantidas: {zeros_mantidos:,}")
    print(f"Candidatos estatísticos a outlier preservados (IQR): {outliers_iqr_mantidos:,}")
    total_codigos_invalidos = sum(codigos_fora_dominio.values())
    print(f"Respostas fora dos domínios definidos no dicionário: {total_codigos_invalidos:,}")
    for coluna, quantidade in codigos_fora_dominio.items():
        print(f"  {coluna}: {quantidade:,}")
    print(f"Arquivo numérico salvo em: {caminho_saida}")
    return caminho_saida

caminho_base_numerica = exportar_base_numerica()


O arquivo numérico já existe; não foi sobrescrito: /Users/joaoigorandrade/Documents/unifor/cienciadedados/ciencia-de-dados-unifor-projeto-2/dados/processed/DADOS/base_modelagem_mt_numerica.csv


## Particionar a base numérica em desenvolvimento e teste

A divisão já foi criada conforme o projeto (`test_size=0.20`, `random_state=42`) e seus IDs estão em `dados/splits/`. Esta célula usa esses IDs para separar a base numérica completa, mantendo as mesmas pessoas e a mesma divisão. Os arquivos de IDs não são apagados: o enunciado pede que sejam preservados.


In [7]:
def separar_base_numerica_por_ids():
    CAMINHO_BASE_NUMERICA = RAIZ_PROJETO / "dados" / "processed" / "DADOS" / "base_modelagem_mt_numerica.csv"
    CAMINHO_IDS_DESENVOLVIMENTO = RAIZ_PROJETO / "dados" / "splits" / "desenvolvimento_ids.csv"
    CAMINHO_IDS_TESTE = RAIZ_PROJETO / "dados" / "splits" / "teste_ids.csv"
    CAMINHO_DESENVOLVIMENTO_NUMERICO = RAIZ_PROJETO / "dados" / "processed" / "DADOS" / "desenvolvimento_80_numerico.csv"
    CAMINHO_TESTE_NUMERICO = RAIZ_PROJETO / "dados" / "processed" / "DADOS" / "teste_20_numerico.csv"

    if not CAMINHO_BASE_NUMERICA.exists():
        raise FileNotFoundError(f"Base numérica não encontrada: {CAMINHO_BASE_NUMERICA}")
    if not CAMINHO_IDS_DESENVOLVIMENTO.exists() or not CAMINHO_IDS_TESTE.exists():
        raise FileNotFoundError("Os arquivos de IDs de desenvolvimento/teste não foram encontrados em dados/splits/.")
    existe_dev = CAMINHO_DESENVOLVIMENTO_NUMERICO.exists()
    existe_teste = CAMINHO_TESTE_NUMERICO.exists()
    if existe_dev and existe_teste:
        print("Os dois CSVs particionados já existem; nenhum arquivo foi sobrescrito.")
        print(f"Desenvolvimento (80%): {len(pd.read_csv(CAMINHO_IDS_DESENVOLVIMENTO)):,} IDs -> {CAMINHO_DESENVOLVIMENTO_NUMERICO}")
        print(f"Teste (20%): {len(pd.read_csv(CAMINHO_IDS_TESTE)):,} IDs -> {CAMINHO_TESTE_NUMERICO}")
        print("Os CSVs de IDs em dados/splits/ foram preservados.")
        return
    if existe_dev or existe_teste:
        raise FileExistsError("Só uma das saídas existe; confira os arquivos antes de continuar para não criar uma partição incompleta.")

    ids_dev = pd.read_csv(CAMINHO_IDS_DESENVOLVIMENTO, dtype={COLUNA_ID: "int64"})[COLUNA_ID].to_numpy()
    ids_teste = pd.read_csv(CAMINHO_IDS_TESTE, dtype={COLUNA_ID: "int64"})[COLUNA_ID].to_numpy()
    if np.intersect1d(ids_dev, ids_teste).size:
        raise ValueError("Há IDs repetidos entre desenvolvimento e teste.")

    arquivo_dev_tmp = CAMINHO_DESENVOLVIMENTO_NUMERICO.with_name(CAMINHO_DESENVOLVIMENTO_NUMERICO.name + ".tmp")
    arquivo_teste_tmp = CAMINHO_TESTE_NUMERICO.with_name(CAMINHO_TESTE_NUMERICO.name + ".tmp")
    if arquivo_dev_tmp.exists() or arquivo_teste_tmp.exists():
        raise FileExistsError("Há arquivo temporário de partição anterior; verifique antes de continuar.")

    linhas_dev = linhas_teste = linhas_base = 0
    primeiro_dev = primeiro_teste = True
    try:
        for bloco in pd.read_csv(CAMINHO_BASE_NUMERICA, chunksize=TAMANHO_BLOCO):
            linhas_base += len(bloco)
            mascara_dev = bloco[COLUNA_ID].isin(ids_dev)
            mascara_teste = bloco[COLUNA_ID].isin(ids_teste)
            if (mascara_dev & mascara_teste).any():
                raise ValueError("Um mesmo registro foi classificado em desenvolvimento e teste.")
            if (~mascara_dev & ~mascara_teste).any():
                raise ValueError("Há registro numérico sem ID correspondente em nenhum split.")

            bloco_dev = bloco.loc[mascara_dev]
            bloco_teste = bloco.loc[mascara_teste]
            if not bloco_dev.empty:
                bloco_dev.to_csv(arquivo_dev_tmp, mode="w" if primeiro_dev else "a", header=primeiro_dev, index=False)
                linhas_dev += len(bloco_dev)
                primeiro_dev = False
            if not bloco_teste.empty:
                bloco_teste.to_csv(arquivo_teste_tmp, mode="w" if primeiro_teste else "a", header=primeiro_teste, index=False)
                linhas_teste += len(bloco_teste)
                primeiro_teste = False

        if linhas_dev != len(ids_dev) or linhas_teste != len(ids_teste):
            raise ValueError(
                f"Contagens não conferem: dev {linhas_dev:,}/{len(ids_dev):,}; "
                f"teste {linhas_teste:,}/{len(ids_teste):,}."
            )
        arquivo_dev_tmp.replace(CAMINHO_DESENVOLVIMENTO_NUMERICO)
        arquivo_teste_tmp.replace(CAMINHO_TESTE_NUMERICO)
    except Exception:
        for temporario in (arquivo_dev_tmp, arquivo_teste_tmp):
            if temporario.exists():
                temporario.unlink()
        raise

    print(f"Base numérica de origem: {linhas_base:,} registros")
    print(f"Desenvolvimento (80%): {linhas_dev:,} -> {CAMINHO_DESENVOLVIMENTO_NUMERICO}")
    print(f"Teste isolado (20%): {linhas_teste:,} -> {CAMINHO_TESTE_NUMERICO}")
    print(f"Partição respeitada: {linhas_dev / linhas_base:.2%} desenvolvimento / {linhas_teste / linhas_base:.2%} teste")
    print("Os CSVs de IDs em dados/splits/ foram preservados.")

separar_base_numerica_por_ids()


Os dois CSVs particionados já existem; nenhum arquivo foi sobrescrito.
Desenvolvimento (80%): 2,153,941 IDs -> /Users/joaoigorandrade/Documents/unifor/cienciadedados/ciencia-de-dados-unifor-projeto-2/dados/processed/DADOS/desenvolvimento_80_numerico.csv
Teste (20%): 538,486 IDs -> /Users/joaoigorandrade/Documents/unifor/cienciadedados/ciencia-de-dados-unifor-projeto-2/dados/processed/DADOS/teste_20_numerico.csv
Os CSVs de IDs em dados/splits/ foram preservados.
